<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/jiwoo/05_jw_cls_20k_recall_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pymysql
import os, pickle, numpy as np, pandas as pd, torch, pymysql
import torch.nn as nn, torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
# crop zip 로컬 복사 + 압축 해제 (런타임마다 1회)
!cp "/content/drive/MyDrive/ToBigs/2425/Pillot/jiwoo/20k/data/20k_cropped_images_v2.zip" "/content/20k_cropped_images_v2.zip"
!unzip -q -o "/content/20k_cropped_images_v2.zip" -d "/content/"
print(len(os.listdir('/content/20k_cropped_images_v2/')))   # 이미지 개수 확인

34021


In [ ]:
# ========== 평가: 후보 압축률 / 정답 포함률 (추론 전용) ==========
# ---------- 0. 경로 ----------
BASE_DIR = '/content/drive/MyDrive/ToBigs/2425/Pillot/jiwoo/20k/'
CKPT     = os.path.join(BASE_DIR, 'checkpoints', 'best_20k_v2_1_redsampler5.pth')
LE_PATH  = os.path.join(BASE_DIR, 'data', 'label_encoders_20k.pkl')
MANIFEST = f'{BASE_DIR}manifest/manifest_clean_20k_33340.csv'
CROP_DIR = '/content/20k_cropped_images_v2/'
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------- 1. DB 속성 ----------
conn = pymysql.connect(host='103.218.161.72', user='jiwoo_admin',
                       password='1234', database='pilliot_db', port=3306)
db = pd.read_sql("""SELECT item_seq, color_class1 AS db_color, drug_shape AS db_shape
                    FROM drug_master
                    WHERE color_class1 IS NOT NULL AND drug_shape IS NOT NULL""", conn)
conn.close()

# ---------- 2. 색 정규화 & 후보 집합 ----------
def normalize_color(c):
    if pd.isna(c): return None
    return str(c).split(',')[0].strip()
db['db_color_norm'] = db['db_color'].apply(normalize_color)
combo_to_items = (db.groupby(['db_color_norm','db_shape'])['item_seq'].apply(set).to_dict())
print(f"DB 조합 수: {len(combo_to_items)} | DB 약 종: {db['item_seq'].nunique()}")

# ---------- 3. 라벨 인코더 ----------
with open(LE_PATH, 'rb') as f:
    enc = pickle.load(f)
le_color, le_shape = enc['color'], enc['shape']

print("color classes:", list(le_color.classes_))
print("shape classes:", list(le_shape.classes_))

COLOR_NAMES = list(le_color.classes_)
SHAPE_NAMES = list(le_shape.classes_)

# ---------- 4. transform / dataset (추론용만) ----------
class LetterboxResize:
    def __init__(self, size=224, fill=128): self.size, self.fill = size, fill
    def __call__(self, img):
        w,h = img.size; s = self.size/max(w,h)
        nw,nh = int(w*s), int(h*s)
        img = TF.resize(img,(nh,nw))
        pw,ph = self.size-nw, self.size-nh
        return TF.pad(img,(pw//2,ph//2,pw-pw//2,ph-ph//2),fill=self.fill)

val_transform = transforms.Compose([
    LetterboxResize(224,128), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class PillDataset(Dataset):
    def __init__(self, df, crop_dir, transform):
        self.df=df.reset_index(drop=True); self.crop_dir=crop_dir; self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=Image.open(os.path.join(self.crop_dir,f"{row['object_id']}.png")).convert('RGB')
        return self.transform(img), {
            'shape':torch.tensor(row['shape_id'],dtype=torch.long),
            'color':torch.tensor(row['color_id'],dtype=torch.long)}

# ---------- 5. val df (crop 필터 후 — 순서 보존의 핵심) ----------
df = pd.read_csv(MANIFEST, low_memory=False)

# 문자열 라벨 → id 컬럼 생성 (인코더의 classes_ 순서 그대로)
df['color_id'] = le_color.transform(df['color_group_normalized'])
df['shape_id'] = le_shape.transform(df['shape_group_unified'])

# 검증 — id↔이름 매핑 눈으로 확인 (하드코딩 금지, 항상 여기서 확인)
print("\ncolor id 매핑:")
for i, c in enumerate(le_color.classes_):
    print(f"  {i}: {c}")
print(f"\ncolor_id 분포: {df['color_id'].value_counts().sort_index().to_dict()}")

df_val = df[df['split']=='val'].reset_index(drop=True)
saved = set(os.listdir(CROP_DIR))
df_val = df_val[df_val['object_id'].astype(str).add('.png').isin(saved)].reset_index(drop=True)
print(f"val: {len(df_val)} | item_seq 종: {df_val['item_seq'].nunique()}")

val_dataset = PillDataset(df_val, CROP_DIR, val_transform)
val_loader  = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ---------- 6. 모델 로드 & 추론 ----------
backbone = models.convnext_tiny(weights=None); backbone.classifier = nn.Identity()
class PillClassifier(nn.Module):
    def __init__(self, backbone, num_classes, feature_dim=768):
        super().__init__(); self.backbone=backbone
        self.heads=nn.ModuleDict({n:nn.Sequential(nn.Dropout(0.3),nn.Linear(feature_dim,c))
                                  for n,c in num_classes.items()})
    def forward(self,x):
        f=self.backbone(x).flatten(1)
        return {n:h(f) for n,h in self.heads.items()}
model = PillClassifier(backbone, {'shape':4,'color':10}).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()

pred_c, pred_s = [], []
with torch.no_grad():
    for imgs,_ in val_loader:
        out = model(imgs.to(device))
        pred_c.extend(out['color'].argmax(1).cpu().numpy())
        pred_s.extend(out['shape'].argmax(1).cpu().numpy())

df_val = df_val.iloc[:len(pred_c)].copy()
df_val['pred_color'] = [COLOR_NAMES[i] for i in pred_c]
df_val['pred_shape'] = [SHAPE_NAMES[i] for i in pred_s]

# ---------- 7. 후보 검색 & 지표 ----------
cand, hit = [], []
for _,r in df_val.iterrows():
    c = combo_to_items.get((r['pred_color'],r['pred_shape']), set())
    cand.append(len(c)); hit.append(r['item_seq'] in c)
df_val['cand_size']=cand; df_val['hit']=hit

print(f"\n평균 후보: {np.mean(cand):.1f}종 | 중앙값 {np.median(cand):.0f} | 최대 {np.max(cand)}")
print(f"recall(정답 후보 포함): {np.mean(hit)*100:.1f}%")
print(f"압축: {db['item_seq'].nunique()} → 평균 {np.mean(cand):.0f}종")

small = df_val[df_val['cand_size']<=5]
print(f"\n[≤5종 범위] {len(small)}장 ({len(small)/len(df_val)*100:.1f}%) | "
      f"평균 {small['cand_size'].mean():.1f}종 | recall {small['hit'].mean()*100:.1f}%")

/tmp/ipykernel_1063/2572788625.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db = pd.read_sql("""SELECT item_seq, color_class1 AS db_color, drug_shape AS db_shape


DB 조합 수: 86 | DB 약 종: 4461
color classes: ['갈색', '기타', '노랑', '분홍', '빨강', '연두', '주황', '초록', '파랑', '하양']
shape classes: ['기타', '원형', '장방형', '타원형']

color id 매핑:
  0: 갈색
  1: 기타
  2: 노랑
  3: 분홍
  4: 빨강
  5: 연두
  6: 주황
  7: 초록
  8: 파랑
  9: 하양

color_id 분포: {0: 2279, 1: 900, 2: 5055, 3: 4386, 4: 1169, 5: 1429, 6: 2394, 7: 2020, 8: 2172, 9: 11536}
val: 7476 | item_seq 종: 1837

평균 후보: 290.7종 | 중앙값 169 | 최대 931
recall(정답 후보 포함): 65.1%
압축: 4461 → 평균 291종

[≤5종 범위] 394장 (5.3%) | 평균 0.5종 | recall 1.8%


In [ ]:
# 후보 0종 케이스 분석
zero = df_val[df_val['cand_size']==0]
print(f"후보 0종: {len(zero)}장 ({len(zero)/len(df_val)*100:.1f}%)")
print(f"  그중 색예측='기타': {(zero['pred_color']=='기타').sum()}장")
print(f"  0종의 예측색 분포:\n{zero['pred_color'].value_counts().to_string()}")

# 0종 제외하고 다시
nonzero = df_val[df_val['cand_size']>=1]
print(f"\n[후보 1종 이상 = 매칭 성공] {len(nonzero)}장 ({len(nonzero)/len(df_val)*100:.1f}%)")
print(f"  평균 후보: {nonzero['cand_size'].mean():.1f}종 | recall: {nonzero['hit'].mean()*100:.1f}%")

# ≤5 범위도 0종 제외하고
small = df_val[(df_val['cand_size']>=1)&(df_val['cand_size']<=5)]
print(f"\n[1~5종 범위, 진짜 잘 좁혀진] {len(small)}장 ({len(small)/len(df_val)*100:.1f}%)")
print(f"  평균: {small['cand_size'].mean():.1f}종 | recall: {small['hit'].mean()*100:.1f}%")

# '기타' 제외 전체 recall (순수 성능)
no_etc = df_val[df_val['pred_color']!='기타']
print(f"\n['기타' 예측 제외] {len(no_etc)}장 | recall: {no_etc['hit'].mean()*100:.1f}%")

후보 0종: 305장 (4.1%)
  그중 색예측='기타': 305장
  0종의 예측색 분포:
pred_color
기타    305

[후보 1종 이상 = 매칭 성공] 7171장 (95.9%)
  평균 후보: 303.0종 | recall: 67.8%

[1~5종 범위, 진짜 잘 좁혀진] 89장 (1.2%)
  평균: 2.1종 | recall: 7.9%

['기타' 예측 제외] 7171장 | recall: 67.8%


In [ ]:
# 색만 맞으면 / 모양만 맞으면 / 둘 다 맞으면 recall 어떻게 갈리나
# 정답 약의 실제 색·모양 (DB 기준)
db_attr = db.set_index('item_seq')[['db_color_norm','db_shape']].to_dict('index')

correct_color = correct_shape = both = 0
for _, r in df_val.iterrows():
    seq = r['item_seq']
    if seq not in db_attr: continue
    true_c = db_attr[seq]['db_color_norm']
    true_s = db_attr[seq]['db_shape']
    cc = (r['pred_color']==true_c)
    cs = (r['pred_shape']==true_s)
    correct_color += cc; correct_shape += cs; both += (cc and cs)
n = len(df_val)
print(f"색 정확도(DB기준): {correct_color/n*100:.1f}%")
print(f"모양 정확도(DB기준): {correct_shape/n*100:.1f}%")
print(f"색+모양 동시: {both/n*100:.1f}%  ← 이게 recall 상한")

색 정확도(DB기준): 78.5%
모양 정확도(DB기준): 83.9%
색+모양 동시: 65.1%  ← 이게 recall 상한


In [ ]:
# 색별 정확도 — 어느 색이 데모에 적합한가
df_val['true_color'] = df_val['item_seq'].map(
    lambda s: db_attr[s]['db_color_norm'] if s in db_attr else None)
df_val['color_ok'] = df_val['pred_color']==df_val['true_color']
print("색별 정확도 (DB기준):")
print(df_val.groupby('true_color')['color_ok'].agg(['mean','count']).sort_values('mean',ascending=False))

색별 정확도 (DB기준):
                mean  count
true_color                 
빨강          0.962898    566
초록          0.962282   1034
연두          0.869318    352
하양          0.851557   3018
파랑          0.751969    254
분홍          0.671329    572
갈색          0.633166    796
노랑          0.613636    528
주황          0.247368    190
보라          0.000000     71
검정          0.000000      4
남색          0.000000      8
청록          0.000000     31
자주          0.000000      9
투명          0.000000     19
회색          0.000000     24


In [ ]:
# 데모 풀(정확도 높은 색) + 색 top-2 허용 시 recall
DEMO_COLORS = ['빨강','초록','연두','하양','파랑']  # 정확도 75%+ 색

# val 약의 true 색이 데모색인 것만
demo = df_val[df_val['true_color'].isin(DEMO_COLORS)].copy()
print(f"데모 풀: {len(demo)}장 ({len(demo)/len(df_val)*100:.1f}%) | item_seq {demo['item_seq'].nunique()}종")
print(f"  현재 recall(top-1): {demo['hit'].mean()*100:.1f}%")

# top-2 색은 logit 필요 — 재추론 시 out['color'].topk(2) 저장해서 비교
# 개념: 색 top-2면 색 정확도 78.5% → ~90%로 오를 것, recall도 그만큼

데모 풀: 5224장 (69.9%) | item_seq 1061종
  현재 recall(top-1): 76.2%


In [ ]:
# 데모 풀 약(1061종)이 DB에서 각인 있는지 + 후보 크기 확인
demo_seqs = demo['item_seq'].unique()
# DB에서 이 약들의 각인·후보크기
conn = pymysql.connect(host='103.218.161.72', user='jiwoo_admin',
                       password='1234', database='pilliot_db', port=3306)
ph = ','.join(['%s']*len(demo_seqs))
dd = pd.read_sql(f"""
    SELECT item_seq,
           ((print_front IS NOT NULL AND print_front<>'') OR
            (print_back IS NOT NULL AND print_back<>'')) AS has_print
    FROM drug_master WHERE item_seq IN ({ph})
""", conn, params=list(demo_seqs))
conn.close()

print(f"데모 풀 {len(demo_seqs)}종 중 각인 있음: {dd['has_print'].sum()}종 "
      f"({dd['has_print'].mean()*100:.0f}%)")
# 각인 있는 약만 최종 데모 풀

/tmp/ipykernel_1063/91734689.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dd = pd.read_sql(f"""


데모 풀 1061종 중 각인 있음: 1058종 (100%)


In [ ]:
# 색 등급별로 풀 넓혀가며 recall 변화
for thr_colors in [
    ['빨강','초록','연두','하양','파랑'],                      # 75%+ (현재)
    ['빨강','초록','연두','하양','파랑','분홍'],               # +분홍
    ['빨강','초록','연두','하양','파랑','분홍','갈색','노랑'],  # +난색대
]:
    sub = df_val[df_val['true_color'].isin(thr_colors)]
    print(f"{len(thr_colors)}색: {len(sub)}장 ({len(sub)/len(df_val)*100:.0f}%) | "
          f"recall {sub['hit'].mean()*100:.1f}% | {sub['item_seq'].nunique()}종")

5색: 5224장 (70%) | recall 76.2% | 1061종
6색: 5796장 (78%) | recall 71.2% | 1249종
8색: 7120장 (95%) | recall 67.8% | 1683종


In [ ]:
import torch.nn.functional as F

# ---------- 재추론: 색·모양 softmax 확률 전체 저장 ----------
color_probs, shape_probs = [], []
with torch.no_grad():
    for imgs, _ in val_loader:        # ← _ 추가 (라벨 무시)
        out = model(imgs.to(device))
        color_probs.append(F.softmax(out['color'], dim=1).cpu())
        shape_probs.append(F.softmax(out['shape'], dim=1).cpu())
color_probs = torch.cat(color_probs).numpy()
shape_probs = torch.cat(shape_probs).numpy()
df_val = df_val.iloc[:len(color_probs)].copy()

# ---------- DB 약마다 (색,모양) 인덱스 매핑 ----------
# 각 DB 약의 정규화색·모양을 모델 클래스 인덱스로
color_to_idx = {c:i for i,c in enumerate(COLOR_NAMES)}
shape_to_idx = {s:i for i,s in enumerate(SHAPE_NAMES)}

db2 = db.copy()
db2['cidx'] = db2['db_color_norm'].map(color_to_idx)   # 기타색은 NaN (모델에 없음)
db2['sidx'] = db2['db_shape'].map(shape_to_idx)
db2 = db2.dropna(subset=['cidx','sidx'])
db2['cidx'] = db2['cidx'].astype(int); db2['sidx'] = db2['sidx'].astype(int)

# item_seq → (cidx, sidx)
seq_attr = db2.set_index('item_seq')[['cidx','sidx']].to_dict('index')
all_seqs = list(seq_attr.keys())
seq_cidx = np.array([seq_attr[s]['cidx'] for s in all_seqs])
seq_sidx = np.array([seq_attr[s]['sidx'] for s in all_seqs])

# ---------- 각 val 이미지: 모든 DB 약에 점수 매겨 Top-K ----------
# 점수 = P(약의 색) * P(약의 모양)  (곱=결합확률. 합으로 바꾸려면 +)
def recall_at_k(idx_mask, K_list=(1,5,10,20,50)):
    sub = df_val[idx_mask].reset_index(drop=True)
    cp = color_probs[idx_mask.values]; sp = shape_probs[idx_mask.values]
    seq_pos = {s:i for i,s in enumerate(all_seqs)}
    hits = {K:0 for K in K_list}; tot = 0
    for n in range(len(sub)):
        seq = sub.iloc[n]['item_seq']
        if seq not in seq_pos: continue
        tot += 1
        scores = cp[n][seq_cidx] * sp[n][seq_sidx]
        # 정답의 순위 = 정답보다 점수 높은 약 수
        rank = (scores > scores[seq_pos[seq]]).sum()  # 0-indexed 순위
        for K in K_list:
            if rank < K: hits[K] += 1
    return {K: hits[K]/tot*100 for K in K_list}, tot

# 전체
mask_all = pd.Series([True]*len(df_val))
r_all, n = recall_at_k(mask_all)
print(f"[전체 {n}장] recall@K (softmax 점수 랭킹):")
for k,v in r_all.items(): print(f"  @{k}: {v:.1f}%")

# 5색 데모 풀
DEMO = ['빨강','초록','연두','하양','파랑']
mask_demo = df_val['true_color'].isin(DEMO)
r_demo, n2 = recall_at_k(mask_demo)
print(f"\n[5색 풀 {n2}장] recall@K:")
for k,v in r_demo.items(): print(f"  @{k}: {v:.1f}%")

[전체 6837장] recall@K (softmax 점수 랭킹):
  @1: 72.5%
  @5: 72.6%
  @10: 72.8%
  @20: 72.9%
  @50: 75.0%

[5색 풀 5027장] recall@K:
  @1: 80.9%
  @5: 81.0%
  @10: 81.0%
  @20: 81.1%
  @50: 82.5%
